In [1]:
# import libraries
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [7]:
# Load environment variables from .env file
load_dotenv()

# Get API key from environment variables
open_ai_key = os.getenv("SBS_OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()
client.api_key = open_ai_key

JSONL_FILE = "requests.jsonl"
OUTPUT_FILE = "output.jsonl"
VOCAB_FOLDER_PATH = "pali_class/vocab"

In [34]:
keyword = input("Enter a Pali word: ").strip() # Remove extra spaces

def search_pali_in_csv(keyword):
    """Search for a Pali word in all CSV files in the folder."""
    result = {
        "id": -1,
        "pali": "",
        "pos": "",
        "exercise_number": ""
    }

    for filename in os.listdir(VOCAB_FOLDER_PATH):
        if filename.endswith(".csv"): # Only search in CSV files
            file_path = os.path.join(VOCAB_FOLDER_PATH, filename)
            df = pd.read_csv(file_path, dtype=str) 
            match = df[df["pali"] == keyword]

            if not match.empty:
                result["id"] = match["id"].values[0]
                result["pali"] = match["pali"].values[0]
                result["pos"] = match["pos"].values[0]
                
                print("Match found in file:", filename)
                
                number = filename.split("_")[-1].split(".")[0] # Extract the number
                result["exercise_number"] = number

                break

    if result["id"] == -1:
        print("No matches found.")

    return result

result = search_pali_in_csv(keyword)
result

Match found in file: vocab_class_22.csv


{'id': '181', 'pali': 'akāsi 1', 'pos': 'aor', 'exercise_number': '22'}

In [68]:
PROMPT = """
You are an AI assistant specialized in Pali language processing. Your task is to extract relevant Pali sentences based on the given keywords, ensuring grammatical accuracy and meaningful context.

For each keyword:
1. Extract exactly **one** relevant Pali sentence.
2. If multiple sentences exist, **prioritize** those without special markers ('$' or '%'). If unavoidable, use the best option available.
3. Preserve existing 'simpl' markers.
4. Provide the **English translation** of the sentence.
5. Include the **source reference**.
6. If no suitable sentence is found, return a flag for manual review.

### Output Format (JSON):
{
  "pali_sentence": "...",
  "translation": "...",
  "source": "...",
  "flag_review": false
}
If no match is found:
{
  "pali_sentence": null,
  "translation": null,
  "source": null,
  "flag_review": true
}

Ensure the response is structured, accurate, and follows these rules.
"""

In [ ]:
batch = client.batches.retrieve("batch_67b72a2355e88190b372d02ff6c2ea2b")

print(batch)

Batch(id='batch_67b72a2355e88190b372d02ff6c2ea2b', completion_window='24h', created_at=1740057123, endpoint='/v1/chat/completions', input_file_id='file-6zUy6wXzWucTcjqYXBFK9d', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1740057139, error_file_id=None, errors=None, expired_at=None, expires_at=1740143523, failed_at=None, finalizing_at=1740057139, in_progress_at=1740057123, metadata={'description': 'nightly eval job'}, output_file_id='file-Ad9gt5GXLR27Gc6YUZNbb6', request_counts=BatchRequestCounts(completed=3, failed=0, total=3))


In [ ]:
{
    "custom_id": "request-1", 
    "method": "POST", 
    "url": "/v1/chat/completions", 
    "body": {
        "model": "gpt-3.5-turbo-0125", 
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Hello world!"}
        ],
    "max_tokens": 1000}
}

In [ ]:
batch_input_file = client.files.create(
    file=open(JSONL_FILE, "rb"),
    purpose="batch"
)

batch_input_file

FileObject(id='file-VrgKiuMcW9vpiMzH7z1rVF', bytes=775, created_at=1740058861, filename='requests.jsonl', object='file', purpose='batch', status='processed', status_details=None, expires_at=None)

In [ ]:
batch_input_file_id = batch_input_file.id

client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="1h",
    metadata={"description": "Automated batch processing"}
)

Batch(id='batch_67b72a2355e88190b372d02ff6c2ea2b', completion_window='24h', created_at=1740057123, endpoint='/v1/chat/completions', input_file_id='file-6zUy6wXzWucTcjqYXBFK9d', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1740143523, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'nightly eval job'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [56]:
batch = client.batches.retrieve("batch_67b72a2355e88190b372d02ff6c2ea2b")

print(batch)

Batch(id='batch_67b72a2355e88190b372d02ff6c2ea2b', completion_window='24h', created_at=1740057123, endpoint='/v1/chat/completions', input_file_id='file-6zUy6wXzWucTcjqYXBFK9d', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1740057139, error_file_id=None, errors=None, expired_at=None, expires_at=1740143523, failed_at=None, finalizing_at=1740057139, in_progress_at=1740057123, metadata={'description': 'nightly eval job'}, output_file_id='file-Ad9gt5GXLR27Gc6YUZNbb6', request_counts=BatchRequestCounts(completed=3, failed=0, total=3))


In [57]:
file_response = client.files.content("file-Ad9gt5GXLR27Gc6YUZNbb6")

print(file_response.text)

{"id": "batch_req_67b72a3349708190b6bd0745acc7aed3", "custom_id": "request-1", "response": {"status_code": 200, "request_id": "bf1e43971fc700b2066d8ec738a66ef7", "body": {"id": "chatcmpl-B30dcFsdcVDccOXUwzLjRHmOb7zZO", "object": "chat.completion", "created": 1740057136, "model": "gpt-3.5-turbo-0125", "choices": [{"index": 0, "message": {"role": "assistant", "content": "Hello! How can I assist you today?", "refusal": null}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 20, "completion_tokens": 10, "total_tokens": 30, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion_tokens_details": {"reasoning_tokens": 0, "audio_tokens": 0, "accepted_prediction_tokens": 0, "rejected_prediction_tokens": 0}}, "service_tier": "default", "system_fingerprint": null}}, "error": null}
{"id": "batch_req_67b72a335de48190a3ff0973d0003f56", "custom_id": "request-2", "response": {"status_code": 200, "request_id": "3d8586029e89ade616f0b31e83ee9824", "body": {"id